# Customer Churn & Retention — ML\n\n## Business question\nWhich customers are most likely to churn, and how can a retention team prioritize them?\n\nThis notebook focuses on **risk ranking and business use**, not on maximizing accuracy. We compare an interpretable Logistic Regression baseline with a Random Forest model, then inspect threshold trade-offs and the highest-risk segment.

## 1. Imports and configuration

In [ ]:
from pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    average_precision_score, confusion_matrix, ConfusionMatrixDisplay,\n    f1_score, precision_score, recall_score, roc_auc_score,\n)\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\n\nRANDOM_STATE = 42\nDATA_PATH = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
df = pd.read_csv(DATA_PATH)\ndf['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')\ndf['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})\n\nprint('Shape:', df.shape)\nprint('Duplicate rows:', df.duplicated().sum())\nprint('Missing values:', int(df.isna().sum().sum()))\nprint('Churn rate:', round(df['Churn'].mean(), 3))

## 2. First look at the target\n\nChurn is the minority class, so accuracy alone would hide an important part of model quality.

In [ ]:
target_share = df['Churn'].value_counts(normalize=True).sort_index()\nprint(target_share.rename({0:'Stayed', 1:'Churned'}))\n\nsns.countplot(data=df, x='Churn')\nplt.title('Customer churn distribution')\nplt.xlabel('Churn (0 = No, 1 = Yes)')\nplt.ylabel('Customers')\nplt.show()

## 3. Prepare the modeling dataset\n\n`customerID` is an identifier, so it is excluded. `TotalCharges` is converted to numeric before the split; missing values are handled inside the preprocessing pipeline.\n\nAll transformations are fitted only on the training sample, which prevents preprocessing leakage.

In [ ]:
X = df.drop(columns=['customerID', 'Churn'])\ny = df['Churn'].astype(int)\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE\n)\n\ncategorical = X.select_dtypes(include=['object']).columns.tolist()\nnumeric = X.select_dtypes(exclude=['object']).columns.tolist()\n\nnumeric_pipe = Pipeline([\n    ('imputer', SimpleImputer(strategy='median')),\n    ('scaler', StandardScaler()),\n])\ncategorical_pipe = Pipeline([\n    ('imputer', SimpleImputer(strategy='most_frequent')),\n    ('onehot', OneHotEncoder(handle_unknown='ignore')),\n])\n\npreprocessor = ColumnTransformer([\n    ('num', numeric_pipe, numeric),\n    ('cat', categorical_pipe, categorical),\n])

## 4. Baseline vs stronger model

In [ ]:
logistic = Pipeline([\n    ('preprocessor', preprocessor),\n    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),\n])\n\nrf_preprocessor = ColumnTransformer([\n    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric),\n    ('cat', Pipeline([\n        ('imputer', SimpleImputer(strategy='most_frequent')),\n        ('onehot', OneHotEncoder(handle_unknown='ignore')),\n    ]), categorical),\n])\n\nrandom_forest = Pipeline([\n    ('preprocessor', rf_preprocessor),\n    ('model', RandomForestClassifier(\n        n_estimators=500, min_samples_leaf=5, class_weight='balanced',\n        random_state=RANDOM_STATE, n_jobs=-1\n    )),\n])\n\nmodels = {'Logistic Regression': logistic, 'Random Forest': random_forest}\nprobabilities = {}\nresults = []\n\nfor name, model in models.items():\n    model.fit(X_train, y_train)\n    p = model.predict_proba(X_test)[:, 1]\n    probabilities[name] = p\n    results.append({\n        'model': name,\n        'ROC-AUC': roc_auc_score(y_test, p),\n        'PR-AUC': average_precision_score(y_test, p),\n    })\n\nresults_df = pd.DataFrame(results).sort_values('PR-AUC', ascending=False)\nresults_df

## 5. Threshold is a business decision\n\nA 0.5 cutoff is arbitrary. The table below shows what happens when we change the operating point. Lower thresholds catch more churners but also flag more customers who would stay.

In [ ]:
best_model_name = results_df.iloc[0]['model']\np = probabilities[best_model_name]\n\nrows = []\nactual_churners = int(y_test.sum())\nfor threshold in np.arange(0.20, 0.81, 0.05):\n    pred = (p >= threshold).astype(int)\n    flagged = int(pred.sum())\n    captured = int(((pred == 1) & (y_test.to_numpy() == 1)).sum())\n    rows.append({\n        'threshold': round(float(threshold), 2),\n        'flagged_customers': flagged,\n        'share_flagged': flagged / len(y_test),\n        'precision': precision_score(y_test, pred, zero_division=0),\n        'recall': recall_score(y_test, pred, zero_division=0),\n        'f1': f1_score(y_test, pred, zero_division=0),\n        'churners_captured': captured,\n        'share_of_churners_captured': captured / actual_churners,\n    })\n\nthresholds_df = pd.DataFrame(rows)\nthresholds_df.round(3)

In [ ]:
plt.figure(figsize=(8, 5))\nplt.plot(thresholds_df['threshold'], thresholds_df['precision'], marker='o', label='Precision')\nplt.plot(thresholds_df['threshold'], thresholds_df['recall'], marker='o', label='Recall')\nplt.xlabel('Decision threshold')\nplt.ylabel('Metric')\nplt.title(f'Threshold trade-off — {best_model_name}')\nplt.legend()\nplt.grid(alpha=0.2)\nplt.show()

## 6. Top-risk targeting\n\nFor a retention team, a ranked list can be more useful than a hard Yes/No prediction. We therefore inspect how many actual churners are concentrated among the highest-risk customers.

In [ ]:
ranked = pd.DataFrame({'actual_churn': y_test.to_numpy(), 'churn_probability': p})\nranked = ranked.sort_values('churn_probability', ascending=False).reset_index(drop=True)\n\nrows = []\nfor share in [0.05, 0.10, 0.20, 0.30, 0.50]:\n    n = max(1, int(np.ceil(len(ranked) * share)))\n    captured = ranked.head(n)['actual_churn'].sum()\n    rows.append({\n        'top_share': share,\n        'customers_targeted': n,\n        'churners_captured': int(captured),\n        'share_of_all_churners_captured': captured / ranked['actual_churn'].sum(),\n    })\n\npd.DataFrame(rows).round(3)

## 7. Final interpretation\n\nThe model is useful only if its ranking supports a realistic retention process. The next business step would be to choose a campaign capacity (for example, the number of customers a team can contact) and select an operating point accordingly.\n\n**Important:** predictive risk is not the same as causal treatment effect. A high-risk customer is not automatically a customer who can be saved. The next experiment should test whether a retention intervention actually reduces churn.